# 01 · NumPy

NumPy (**Num**erical **Py**thon) lets us store numbers in arrays and calculate
with many values at once. We will start with small arrays we can read, then
work with four illustrative flower measurements.

By the end, you should be able to **create and reshape an array**, **select
rows and columns**, and **calculate with whole arrays and along an axis**.

**How to use this notebook:** predict → run → explain. Plotting cells are
provided so you can see what the arrays mean; you do not need to memorize
Matplotlib syntax. Run cells from top to bottom in Colab or local Jupyter.
This notebook needs only NumPy and Matplotlib, with no data download.

**In class:** follow sections A–E, including the short vectorization demo.
**After class:** the **Exercises for you** section is for practice at your own pace.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Display settings only: calculations keep their full precision.
np.set_printoptions(precision=3, suppress=True)

## A. From a Python list to an array
A list is a general-purpose Python container. A NumPy array stores elements
with one shared data type (`dtype`) and supports arithmetic on all entries.

**Predict:** what will multiplying each of these objects by `2` do?

In [ ]:
numbers = [1, 2, 3]
a = np.array(numbers)

print("Python list * 2:", numbers * 2)
print("NumPy array * 2:", a * 2)
print("shape:", a.shape, "dimensions:", a.ndim, "size:", a.size, "dtype:", a.dtype)

### A few useful ways to create arrays

| Function | What you choose | Example |
|---|---|---|
| `np.array(...)` | The actual values | `np.array([1, 2, 3])` |
| `np.zeros(...)`, `np.ones(...)` | The shape | `np.zeros((2, 3))` |
| `np.arange(start, stop, step)` | A step size; `stop` is excluded | `np.arange(0, 10, 2)` |
| `np.linspace(start, stop, num)` | A number of points; both ends are included by default | `np.linspace(0, 1, 5)` |

The tuple `(2, 3)` means **2 rows, 3 columns**. With `dtype=int` we explicitly
request integers; zeros and ones otherwise default to floating-point values.

In [ ]:
print("Zeros:\n", np.zeros((2, 3), dtype=int))
print("Ones:", np.ones(3))
print("A step of 2:", np.arange(0, 10, 2))
print("Five evenly spaced points:", np.linspace(0, 1, 5))

### The same values, a different shape
`reshape(3, 4)` arranges 12 values into three rows of four. The number of
elements must stay the same: `3 × 4 = 12`. By default, values fill one row
after another. We will reuse this small grid to understand axes.

**Predict:** could these 12 values be reshaped to `(2, 6)`? What about `(3, 5)`?

In [ ]:
values = np.arange(12)
grid = values.reshape(3, 4)
print("1D:", values, "shape:", values.shape)
print("2D:\n", grid)
print("shape:", grid.shape, "dimensions:", grid.ndim, "size:", grid.size)

## B. Select a piece of an array
Indices start at **0**; `-1` selects the last entry. A slice is
`start:stop:step`, with `stop` excluded. A bare `:` keeps everything along
that axis. For a 2D array, write **`array[rows, columns]`**.

**Predict:** which values will `grid[1, 1:3]` keep?

In [ ]:
print("First / last value:", values[0], values[-1])
print("First five:", values[:5])
print("Every second value:", values[::2])
print("Reversed:", values[::-1])
print("First two rows:\n", grid[:2, :])
print("Second row, columns 1 and 2:", grid[1, 1:3])

### A tiny table of flower measurements
Each row below is one illustrative flower; the columns are **petal length**
and **petal width**, both in centimetres. This is a small teaching example,
not a sample for drawing conclusions about flowers.

For machine learning, we often call this array `X` and use the convention
`(n_samples, n_features)`. Here that is `(4, 2)`.

In [ ]:
X = np.array([[1.4, 0.2],
              [4.7, 1.4],
              [4.5, 1.5],
              [6.0, 2.5]])
print(X)
print("shape:", X.shape, "dimensions:", X.ndim, "dtype:", X.dtype)

**A shape detail worth noticing:** an integer index removes an axis, while
a slice keeps it. Predict the shapes before running the next cell.

| Selection | Meaning | Shape |
|---|---|---|
| `X[0]` | One flower's two measurements | `(2,)` |
| `X[:, 0]` | All petal lengths, as a 1D array | `(4,)` |
| `X[:, 0:1]` | All petal lengths, keeping a 2D column | `(4, 1)` |

`X[:, [0]]` also keeps a 2D column. A list of indices selects the named
columns; we will return to copying and slicing in the **Exercises for you** section.

In [ ]:
print("First row:", X[0], "shape:", X[0].shape)
print("First column:", X[:, 0], "shape:", X[:, 0].shape)
print("One column, kept 2D:\n", X[:, 0:1])
print("2D column shape:", X[:, 0:1].shape)

### Ask a question with a Boolean mask
Which flowers have petal length above `4` cm? The comparison produces one
`True` or `False` per row. `X[mask]` keeps the rows marked `True`.

In [ ]:
mask = X[:, 0] > 4
selected = X[mask]
print("Keep this row?", mask)
print("Selected flowers:\n", selected)
print("Selected shape:", selected.shape)

In [ ]:
# Each point uses column 0 as x and column 1 as y.
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.scatter(X[:, 0], X[:, 1], s=100, facecolors="none",
           edgecolors="#555555", label="All four flowers")
ax.scatter(selected[:, 0], selected[:, 1], s=55, marker="^",
           color="#0072B2", label="Kept by the mask")
ax.axvline(4, color="#555555", linestyle="--", label="Length = 4 cm")
ax.set(xlabel="Petal length (cm)", ylabel="Petal width (cm)",
       title="A Boolean mask selects whole rows")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

## C. Calculate with the whole array
**Vectorization** means expressing the calculation as an array operation
instead of writing a Python loop over each entry. Arithmetic such as `* 10`
and functions such as `np.sqrt` operate element by element.

**Predict:** what shape does `X * 10` have? Does it change `X` itself?

In [ ]:
X_mm = X * 10
print("Petal measurements in millimetres:\n", X_mm)
print("Original first row, still in cm:", X[0])
print("Square roots:", np.sqrt(np.array([0, 1, 4, 9])))

### Demo · The speed of vectorization (3 min)
We have just applied `np.sqrt` to a few numbers. What happens with 100,000
values? This demonstration compares a Python loop with a single NumPy expression. Both compute the same square roots.

NumPy performs this numerical operation in a compiled loop, reducing the
overhead of processing each value in Python. Run the provided cell and
compare the times; you do not need to learn the timing code.

We time batches of 10 runs, repeat each batch three times, then report the
fastest batch's average time per run. Timing depends on the machine and
input size, so measure rather than assume a fixed speedup.

In [ ]:
import math
import timeit

timing_values = np.arange(100_000, dtype=float)
list_result = [math.sqrt(x) for x in timing_values]
array_result = np.sqrt(timing_values)
assert np.allclose(list_result, array_result)
for label, statement in [
    ("Python loop", "[math.sqrt(x) for x in timing_values]"),
    ("NumPy", "np.sqrt(timing_values)"),
]:
    elapsed = min(timeit.repeat(statement, globals=globals(), repeat=3, number=10)) / 10
    print(f"{label}: {elapsed:.6f} seconds per run")

### Which way does the calculation go?
A reduction such as `mean` combines entries. `axis` specifies the dimension
being collapsed. Our grid has shape `(3, 4)`:

| Expression | Entries combined | Result shape |
|---|---|---|
| `grid.mean()` | All 12 values | Scalar |
| `grid.mean(axis=0)` | Across rows, one mean per column | `(4,)` |
| `grid.mean(axis=1)` | Across columns, one mean per row | `(3,)` |

**Why does `axis=0` give one mean per column?** In `grid[row, column]`, axis `0` corresponds to the row index. With `mean(axis=0)`, we move through the different rows **while staying in the same column**, then combine their values into a mean.

For example, in the first column, we combine `0`, `4`, and `8`: their mean is `4`. We repeat this for each column, leaving **one mean per column**.

```text
     axis=0: move down ↓
        0   1   2   3
        4   5   6   7
        8   9  10  11
        ↓   ↓   ↓   ↓
mean    4   5   6   7
```

**Remember: reduce the chosen axis; the other dimensions remain.**

- `axis=0`: collapse the rows → one value per column.
- `axis=1`: collapse the columns → one value per row.

With `axis=1`, we stay in the same row and move through the columns. For the first row, `[0, 1, 2, 3]`, the mean is `1.5`. The plots show the same two reductions.

In [ ]:
column_grid_means = grid.mean(axis=0)
row_grid_means = grid.mean(axis=1)
print("Grid:\n", grid)
print("Overall mean:", grid.mean())
print("axis=0:", column_grid_means, "shape:", column_grid_means.shape)
print("axis=1:", row_grid_means, "shape:", row_grid_means.shape)

In [ ]:
# Provided display code: the calculations are in the cell above.
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(grid, cmap="Blues", vmin=0, vmax=15)
for row in range(3):
    for col in range(4):
        axes[0].text(col, row, str(grid[row, col]), ha="center", va="center")
axes[0].set(title="grid: shape (3, 4)", xlabel="Column index",
            ylabel="Row index", xticks=range(4), yticks=range(3))

axes[1].bar(np.arange(4), column_grid_means, color="#0072B2")
axes[1].set(title="axis=0 → shape (4,)", xlabel="Column index",
            ylabel="Mean", xticks=range(4), ylim=(0, 11))
axes[2].bar(np.arange(3), row_grid_means, color="#D55E00")
axes[2].set(title="axis=1 → shape (3,)", xlabel="Row index",
            ylabel="Mean", xticks=range(3), ylim=(0, 11))
fig.tight_layout()
plt.show()

## D. Broadcasting: one mean for each feature
Back to the flowers: `X.mean(axis=0)` gives two numbers, one mean per
feature. Subtracting them from `X` **centres each column**.

```text
X                         column_means           centered
4 rows × 2 columns    −    2 values           →   4 rows × 2 columns
(4, 2)                    (2,)                   (4, 2)
```

NumPy applies the length mean to every length and the width mean to every
width. This is **broadcasting**: compare shapes from the right; dimension
sizes must match or one must be `1`. A missing leading dimension acts like `1`.

**Predict:** what should the two column means be after centring?

In [ ]:
column_means = X.mean(axis=0)
centered = X - column_means
print("Feature means (cm):", column_means)
print("Centered measurements (cm):\n", centered)
print("Centered column means:", centered.mean(axis=0))
assert np.allclose(centered.mean(axis=0), 0)

`np.allclose` checks approximate equality: floating-point calculations can
leave tiny rounding differences. An `assert` that passes produces no output.

A negative centred value means **below that feature's mean**. Centring does
not divide by a standard deviation. Also, averaging different physical
features within each row is not automatically meaningful, even when NumPy
permits it.

### Exercise D · Array practice (7 min)
1. Keep rows whose petal width is at least `1.5` cm.
2. Compute the mean petal length for those rows, without a Python loop.
3. Create `np.arange(12).reshape(3, 4)`. Predict the shapes after
   `mean(axis=0)` and `mean(axis=1)`, then check.

**Checkpoints:** two selected rows, mean length `5.25` cm, reduction shapes
`(4,)` and `(3,)`.

<details>
<summary>Hint</summary>

Petal width is column `1`. Build a Boolean mask using `>=`, select the rows,
then take column `0` of those rows before calling `.mean()`.

</details>

In [ ]:
# 1. Create a width mask and use it to select whole rows from X.

# 2. Compute the selected flowers' mean petal length.

# 3. Reshape the integers 0–11, then print the two mean result shapes.

## E. A small bridge to machine learning
`*` multiplies entries element by element. `@` performs matrix multiplication:
here it multiplies each feature by its weight and adds the results per row.

For `X` of shape `(4, 2)` and weights `w` of shape `(2,)`, `X @ w` returns
four scores, shape `(4,)`. The scalar intercept `b` is added to every score.
These are arbitrary demonstration weights, not a trained model or probabilities.

For the first flower: `1.4 × 0.3 + 0.2 × (−0.2) + 0.1 = 0.48`.

In [ ]:
w = np.array([0.3, -0.2])
b = 0.1
contributions = X * w
scores = X @ w + b
print("Element-wise contributions, shape", contributions.shape, ":\n", contributions)
print("Scores:", scores, "shape:", scores.shape)
assert scores.shape == (4,)
assert np.allclose(scores, contributions.sum(axis=1) + b)

### Pause and explain
What does one row of `X` represent? Why does `X[:, 0]` have shape `(4,)`?
Which axis gives one mean per feature? How does NumPy subtract two feature
means from four flowers?

**This is the end of the guided core.** Use the reference below when
preparing Homework 1, and work through **Exercises for you** after class
at your own pace.

## NumPy tools for Homework 1
Keep this reference nearby while preparing Homework 1. The usual operators
`+`, `-`, `*`, `/` and `**` work element by element on arrays.

| Tool | What it does | Small example or shape rule |
|---|---|---|
| `X.shape`, [`X.T`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.T.html) | Inspect dimensions; transpose a 2D array by swapping rows and columns. | `(4, 2)` becomes `(2, 4)` with `.T`. |
| `X @ v` | Matrix–vector multiplication; `*` multiplies element by element. | `(4, 2) @ (2,)` gives `(4,)`; `(2, 4) @ (4,)` gives `(2,)`. |
| [`np.dot(X, v)`](https://numpy.org/doc/stable/reference/generated/numpy.dot.html) | Same result as `X @ v` for the 2D matrix and 1D vector used here. | `(4, 2) @ (2,)` gives `(4,)`. |
| [`np.matmul(X, v)`](https://numpy.org/doc/stable/reference/generated/numpy.matmul.html) | Function form of `X @ v`. | `(4, 2) @ (2,)` gives `(4,)`. |
| [`np.multiply(X, v)`](https://numpy.org/doc/stable/reference/generated/numpy.multiply.html) | Function form of `X * v`; broadcasts `v` across the rows. | `(4, 2) * (2,)` gives `(4, 2)`. |
| [`np.exp(x)`](https://numpy.org/doc/stable/reference/generated/numpy.exp.html) | Calculate the exponential of each value. | `np.exp([0, 1])` ≈ `[1, 2.718]`. |
| [`np.log(x)`](https://numpy.org/doc/stable/reference/generated/numpy.log.html) | Natural logarithm of each value; use positive inputs for finite real results. | `np.log([1, np.e])` ≈ `[0, 1]`. |
| [`np.maximum(x, a)`](https://numpy.org/doc/stable/reference/generated/numpy.maximum.html) | Compare each value with `a` and keep the larger one. | `np.maximum([-2, 3], 1)` gives `[1, 3]`. |
| `np.max(x)` | Reduce to the largest value, rather than comparing element by element. | `np.max([2, 5, 3])` gives `5`. |
| [`np.sum(x)`](https://numpy.org/doc/stable/reference/generated/numpy.sum.html), `x.sum()` | Add values; `axis` chooses which dimension to reduce. | `np.sum([2, 4, 6])` gives `12`. |
| [`np.clip(x, low, high)`](https://numpy.org/doc/stable/reference/generated/numpy.clip.html) | Limit values to an interval. | `np.clip([-1., 0.4, 2.], 0, 1)` gives `[0., 0.4, 1.]`. |

**Two checks for HW1:** `.T` does not turn a 1D array of shape `(D,)` into a
column; it stays `(D,)`. The assignment's cross-entropy loss is a **total**,
so use a sum rather than a mean, and respect the requested return types.

### Numerical stability, briefly
Floating-point numbers have limited precision and range. A direct formula
can encounter infinities (`inf`) or undefined results (`nan`).

- **Logarithms at zero:** `np.log(0.0)` gives `-inf`. For valid probabilities
  in `[0, 1]`, clipping to `[eps, 1 - eps]` before taking `log(p)` or
  `log(1 - p)` avoids zero arguments. For these `float64` examples,
  `eps = 1e-12` is a small safeguard; it slightly changes extreme probabilities.
- **Exponential overflow:** `np.exp(1000.0)` exceeds the `float64` range.
  For HW1's softmax vector, subtract its largest score before exponentiating:
  `z - np.max(z)`. For finite scores, the exponentials are then at most `1`;
  the normalized probabilities are mathematically unchanged. This shift is
  specific to softmax, not to sigmoid. [Why shifting helps](https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.softmax.html).
- **Rounding:** use `np.allclose(actual, expected)` to compare floating-point
  results approximately, rather than expecting exact equality.


## Exercises for you

These exercises and explorations are for you to work through after class.
Try the tasks, modify the provided examples, and explain what you observe.
Hints and reference solutions are available at the end of the notebook.

### 1. Draw with an array
An image can be a 2D array: each entry controls one square's colour.
`imshow` displays it, with row `0` at the top by default.

Start with a `6 × 6` array of integer zeros. Set the first and last rows,
then the first and last columns, to `1` to draw a border. Four simple slice
assignments are fine. This activity is adapted from NumPy 100 exercise 15.

**Checkpoint:** 20 border entries equal `1`; the `4 × 4` interior stays `0`.
As a tiny warm-up, try making a length-10 zero array with only its fifth
entry set to `1` (NumPy 100 exercise 6).

In [ ]:
border = np.zeros((6, 6), dtype=int)
# Set the four edges to 1 here, then rerun the display cell below.

In [ ]:
# Before you fill the edges, an all-white square is expected.
print(border)
print("Number of ones:", (border == 1).sum())
fig, ax = plt.subplots(figsize=(4, 3.5))
display = ax.imshow(border, cmap="Greys", vmin=0, vmax=1, interpolation="nearest")
ax.set(title="Your array as an image", xlabel="Column index",
       ylabel="Row index", xticks=range(6), yticks=range(6))
fig.colorbar(display, ax=ax, ticks=[0, 1], label="Array value")
fig.tight_layout()
plt.show()

### 2. From numbers to curves and distributions
#### A line plot: one output for every input
`np.linspace` creates evenly spaced angles in **radians**, stored in `x`, and `np.sin(x)` produces one value for
each angle. The plot connects those `(x, y)` pairs in order.

**Try it:** change `n_points` from `100` to `8`, rerun, and compare the curve.
Then restore `100` and try `2 * np.sin(x)`. Explain what changed.

In [ ]:
n_points = 100
x = np.linspace(0, 2 * np.pi, n_points)
y = np.sin(x)
print("x shape:", x.shape, "y shape:", y.shape)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(x, y, ".-", color="#0072B2", label="Sampled sine curve")
ax.set(xlabel="Angle (radians)", ylabel="Function value",
       title=f"A vectorized function at {n_points} points")
ax.axhline(0, color="#777777", linewidth=0.8)
ax.legend()
fig.tight_layout()
plt.show()

#### A histogram: how are the values distributed?
We use `np.random.default_rng(42)` to create a random-number generator with a fixed
seed. Recreating it with the same seed makes this example repeatable in the
same environment.

We draw `1,000` synthetic values from a normal distribution with mean `0`
and standard deviation `1`. The **sample** mean will be near, but usually
not exactly, zero. A histogram counts how many values fall in each interval
(bin); it does not show the order in which they were drawn.

**Try it:** change only `bins` from `20` to `8`. Does the sample mean change?

In [ ]:
rng = np.random.default_rng(42)
samples = rng.normal(loc=0, scale=1, size=1_000)
sample_mean = samples.mean()
print(f"Minimum: {samples.min():.3f}, mean: {sample_mean:.3f}, maximum: {samples.max():.3f}")

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(samples, bins=20, color="#0072B2", edgecolor="white")
ax.axvline(sample_mean, color="#D55E00", linestyle="--",
           label=f"Sample mean = {sample_mean:.3f}")
ax.set(xlabel="Synthetic value", ylabel="Number of observations",
       title="A random array summarized by a histogram")
ax.legend()
fig.tight_layout()
plt.show()

### 3. A slice can change its original array
Basic slicing returns a **view**: it shares data with the original array.
Use `.copy()` if you want to edit a slice independently. Boolean masks and
lists of indices return copies when you read with them. Assignment such as `a[mask] = 0` still updates `a` itself.

**Predict:** which assignment below changes `original`?

In [ ]:
original = np.array([10, 20, 30, 40])
view = original[:2]
independent = original[:2].copy()
view[0] = 99
independent[1] = -1
print("Original:", original)
print("View:", view)
print("Independent copy:", independent)

## Check your work
Try the exercises before opening these solutions. The blank practice cells
do not feed later examples, so you can still run the notebook from the top.
For the border activity, the display intentionally stays blank until you
fill its edges.

<details>
<summary>Exercise D · Reference solution</summary>

```python
width_mask = X[:, 1] >= 1.5
wide_flowers = X[width_mask]
mean_length = wide_flowers[:, 0].mean()
practice_grid = np.arange(12).reshape(3, 4)

print(wide_flowers)
print("Mean petal length:", mean_length)
print("axis=0 shape:", practice_grid.mean(axis=0).shape)
print("axis=1 shape:", practice_grid.mean(axis=1).shape)

assert wide_flowers.shape == (2, 2)
assert np.isclose(mean_length, 5.25)
assert practice_grid.mean(axis=0).shape == (4,)
assert practice_grid.mean(axis=1).shape == (3,)
```

The selected rows are `[4.5, 1.5]` and `[6.0, 2.5]`. Their mean petal
length is `(4.5 + 6.0) / 2 = 5.25` cm.

</details>

<details>
<summary>Border and warm-up · Reference solution</summary>

```python
border = np.zeros((6, 6), dtype=int)
border[0, :] = 1
border[-1, :] = 1
border[:, 0] = 1
border[:, -1] = 1
assert border.sum() == 20
assert np.all(border[1:-1, 1:-1] == 0)

warmup = np.zeros(10, dtype=int)
warmup[4] = 1  # The fifth entry has index 4.
assert warmup.sum() == 1 and warmup[4] == 1
```

Rerun the border display cell to see the frame. `1:-1` excludes the first
and last entry along an axis. Each corner belongs to a row and a column,
but assigning `1` twice does not add another `1`.

</details>

<details>
<summary>Predictions and plot activities · What to notice</summary>

- A Python list repeats with `* 2`; the NumPy array doubles each value.
- `(2, 6)` holds 12 values, but `(3, 5)` would need 15.
- `grid[1, 1:3]` is `[5, 6]`; the stop index `3` is excluded.
- `X * 10` keeps shape `(4, 2)` and creates a new result.
- Centring makes each column mean approximately zero.
- Eight points produce a visibly coarser sine curve. Multiplying by `2`
  doubles its vertical amplitude without changing its period.
- Changing histogram bins groups the same samples differently; it does
  not change their mean.
- Editing `view` changes `original` to `[99, 20, 30, 40]`. Editing
  `independent` does not change the original.

</details>

## Sources and further practice

- [Nicolas P. Rougier's NumPy 100](https://github.com/rougier/numpy-100):
  beginner inspiration from exercises **6** (set one value), **8** (reverse),
  **9** (reshape), **13–14** (min/max/mean), and **15** (border). Examples here use our own values and wording.

Reference: [NumPy quickstart](https://numpy.org/doc/stable/user/quickstart.html),
[indexing and views](https://numpy.org/doc/stable/user/basics.indexing.html),
[random generators](https://numpy.org/doc/stable/reference/random/generator.html),
and [Matplotlib's array display](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html).
Reading: *Python Data Science Handbook*, Introduction to NumPy.